In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import itertools
import time

import reservoirpy
from reservoirpy.mat_gen import uniform
from reservoirpy.nodes import Reservoir, Ridge, ScikitLearnNode
from sklearn.linear_model import Lasso

reservoirpy.set_seed(42)

import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..',)))
from lib.utils.reservoirpy import *

# ==========================================================
# LOAD DATA
# ==========================================================
dataset = np.loadtxt('../../data/chaotic_data/lorenz_system.csv', delimiter=',')

# Expecting shape (T, 3) with columns = (x, y, z).
# If your CSV has an extra time column, do: dataset = dataset[:, 1:4]
assert dataset.ndim == 2 and dataset.shape[1] >= 3, \
    f"Expected (T, >=3) Lorenz array, got {dataset.shape}"
dataset = dataset[:, :3]

dim_names = ['x', 'y', 'z']

# Visualize the three components
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for d in range(3):
    axes[d].plot(dataset[:, d], color='steelblue', linewidth=0.8)
    axes[d].set_ylabel(dim_names[d])
axes[-1].set_xlabel('Step')
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# DATA PREPARATION
# ==========================================================
# 3-D Lorenz: data shape is already (T, 3)
data = dataset

# One-step supervised pairs (input -> next-step state)
X_raw = data[:-1]   # shape (T-1, 3)
Y_raw = data[1:]    # shape (T-1, 3)


# ==========================================================
# FIXED SETTINGS
# ==========================================================
train_len = 10000
test_start = 10000
test_len = 10000

regression_model = "ridge"
seed = 42

X_train_raw = X_raw[:train_len]
Y_train_raw = Y_raw[:train_len]

X_test_raw = X_raw[test_start:test_start + test_len]
Y_test_raw = Y_raw[test_start:test_start + test_len]

# Output dimensionality (Lorenz = 3)
n_dim = X_train_raw.shape[1]


# ==========================================================
# PARAMETER GRID
# ==========================================================
param_grid = {
    "normalization":          ["zscore"],
    "train_warmup":           [500],
    "test_warmup":            [500],
    "units":                  [500, 1000, 1500],
    "reservoir_connectivity": [0.2, 0.5, 0.8, 1.0],
    "sr":                     [0.8, 0.9, 0.95, 1.0, 1.2, 1.3],
    "lr":                     [0.1, 0.3, 0.5],
    "input_scaling":          [0.5, 0.8, 1.0],
    "regression":             [1e-8, 1e-6, 1e-5],
}

keys = list(param_grid.keys())
combos = list(itertools.product(*[param_grid[k] for k in keys]))

print(f"Total combinations to evaluate: {len(combos)}")


# ==========================================================
# EVALUATION FUNCTION (3-D)
# ==========================================================
def evaluate_esn(
    normalization,
    train_warmup,
    test_warmup,
    units,
    reservoir_connectivity,
    sr,
    lr,
    input_scaling,
    regression,
    seed=42
):
    """
    Build, train, synchronize, and run closed-loop prediction on 3-D Lorenz.
    Returns NRMSE in scaled space, averaged over the 3 components.
    """
    try:
        if test_warmup >= test_len:
            return np.inf

        # -------------------------
        # Fit scaler on training inputs only (per-dimension)
        # -------------------------
        scaler = fit_scaler(X_train_raw, method=normalization)

        X_train = transform_array(X_train_raw, scaler)
        Y_train = transform_array(Y_train_raw, scaler)
        X_test  = transform_array(X_test_raw,  scaler)
        Y_test  = transform_array(Y_test_raw,  scaler)

        pred_len = test_len - test_warmup
        Y_true_scaled = Y_test[test_warmup:test_warmup + pred_len, :]   # (pred_len, 3)

        if train_warmup >= len(X_train):
            return np.inf

        # -------------------------
        # Reservoir
        # -------------------------
        reservoir = Reservoir(
            units=int(units),
            lr=lr,
            W=uniform(low=-0.5, high=0.5,
                      connectivity=reservoir_connectivity, sr=sr),
            Win=uniform(low=-0.5 * input_scaling, high=0.5 * input_scaling,
                        connectivity=1.0),
            bias=uniform(low=-0.5, high=0.5, connectivity=1.0),
            activation="tanh",
            seed=seed
        )

        if regression_model == "ridge":
            readout = Ridge(ridge=regression)
        else:
            readout = ScikitLearnNode(model=Lasso, alpha=regression,
                                      max_iter=50000, tol=1e-3)

        esn = reservoir >> readout

        # -------------------------
        # Train: reservoirpy automatically handles 3-D input/output
        # -------------------------
        esn.fit(X_train, Y_train, warmup=train_warmup)

        # -------------------------
        # Test: reset + synchronize + closed loop
        # -------------------------
        esn.reset()

        if test_warmup > 0:
            esn.run(X_test[:test_warmup])

        # Start closed loop from the first test point after warmup
        current_input = X_test[test_warmup:test_warmup + 1]   # shape (1, 3)
        Y_pred_scaled = np.zeros((pred_len, n_dim))

        for t in range(pred_len):
            pred = esn.run(current_input)
            pred = np.asarray(pred).reshape(1, n_dim)
            Y_pred_scaled[t, :] = pred[0, :]
            current_input = pred   # feed prediction back as next input

        # Divergence check
        if np.any(np.isnan(Y_pred_scaled)) or np.any(np.isinf(Y_pred_scaled)):
            return np.inf
        if np.max(np.abs(Y_pred_scaled)) > 1e6:
            return np.inf

        # NRMSE per-dimension, then average
        rmse_per_dim = np.sqrt(np.mean((Y_true_scaled - Y_pred_scaled) ** 2, axis=0))
        std_per_dim  = np.std(Y_true_scaled, axis=0)
        if np.any(std_per_dim == 0):
            return np.inf

        nrmse_per_dim = rmse_per_dim / std_per_dim
        return float(np.mean(nrmse_per_dim))

    except Exception as e:
        print(f"    ERROR: {e}")
        return np.inf


# ==========================================================
# RUN GRID SEARCH
# ==========================================================
results = []
best_nrmse = np.inf
best_params = None

t_start = time.time()

for i, combo in enumerate(combos):
    params = dict(zip(keys, combo))
    nrmse = evaluate_esn(**params, seed=seed)
    results.append({**params, "nrmse": nrmse})

    if nrmse < best_nrmse:
        best_nrmse = nrmse
        best_params = params.copy()

    if (i + 1) % 20 == 0 or (i + 1) == len(combos):
        elapsed = time.time() - t_start
        print(f"[{i+1}/{len(combos)}]  elapsed: {elapsed:.0f}s  "
              f"current best NRMSE: {best_nrmse:.6f}")

total_time = time.time() - t_start


# ==========================================================
# RESULTS
# ==========================================================
print("\n" + "=" * 80)
print("GRID SEARCH COMPLETE (3-D Lorenz)")
print(f"Train interval      : [0 : {train_len}]")
print(f"Test interval       : [{test_start} : {test_start + test_len}]")
print(f"Total time          : {total_time:.1f}s "
      f"({total_time/len(combos):.2f}s per trial)")
print(f"Regression model    : {regression_model}")
print(f"Best mean NRMSE     : {best_nrmse:.6f}")
print("Best parameters:")
for k, v in best_params.items():
    print(f"  {k:24s} = {v}")
print("=" * 80)

results_sorted = sorted(results, key=lambda r: r["nrmse"])

print("\nTop 10 configurations:")
print(f"{'Rank':>4}  {'norm':>8}  {'tr_wu':>5}  {'te_wu':>5}  "
      f"{'units':>5}  {'conn':>5}  {'sr':>4}  {'lr':>4}  "
      f"{'in_sc':>6}  {'reg':>8}  {'NRMSE':>10}")
print("-" * 100)
for rank, r in enumerate(results_sorted[:10], 1):
    print(f"{rank:4d}  {r['normalization']:>8s}  "
          f"{r['train_warmup']:5d}  {r['test_warmup']:5d}  "
          f"{int(r['units']):5d}  {r['reservoir_connectivity']:5.2f}  "
          f"{r['sr']:4.2f}  {r['lr']:4.1f}  {r['input_scaling']:6.2f}  "
          f"{r['regression']:8.1e}  {r['nrmse']:10.6f}")


# ==========================================================
# REBUILD BEST MODEL
# ==========================================================
best_scaler = fit_scaler(X_train_raw, method=best_params["normalization"])

X_train_best = transform_array(X_train_raw, best_scaler)
Y_train_best = transform_array(Y_train_raw, best_scaler)
X_test_best  = transform_array(X_test_raw,  best_scaler)
Y_test_best  = transform_array(Y_test_raw,  best_scaler)

best_pred_len = test_len - best_params["test_warmup"]

best_reservoir = Reservoir(
    units=int(best_params["units"]),
    lr=best_params["lr"],
    W=uniform(low=-0.5, high=0.5,
              connectivity=best_params["reservoir_connectivity"],
              sr=best_params["sr"]),
    Win=uniform(low=-0.5 * best_params["input_scaling"],
                high=0.5 * best_params["input_scaling"],
                connectivity=1.0),
    bias=uniform(low=-0.5, high=0.5, connectivity=1.0),
    activation="tanh",
    seed=seed
)

if regression_model == "ridge":
    best_readout = Ridge(ridge=best_params["regression"])
else:
    best_readout = ScikitLearnNode(model=Lasso,
                                   alpha=best_params["regression"],
                                   max_iter=10000)

best_esn = best_reservoir >> best_readout

best_esn.fit(X_train_best, Y_train_best, warmup=best_params["train_warmup"])

best_esn.reset()
if best_params["test_warmup"] > 0:
    best_esn.run(X_test_best[:best_params["test_warmup"]])

Y_pred_best_scaled = np.zeros((best_pred_len, n_dim))
current_input = X_test_best[
    best_params["test_warmup"]:best_params["test_warmup"] + 1
]

for t in range(best_pred_len):
    pred = best_esn.run(current_input)
    pred = np.asarray(pred).reshape(1, n_dim)
    Y_pred_best_scaled[t, :] = pred[0, :]
    current_input = pred

Y_true_best_scaled = Y_test_best[
    best_params["test_warmup"]:best_params["test_warmup"] + best_pred_len, :
]

# Inverse transform back to original scale (per dimension)
Y_pred_best = inverse_transform_array(Y_pred_best_scaled, best_scaler)
Y_true_best = inverse_transform_array(Y_true_best_scaled, best_scaler)

# Per-dimension and aggregate metrics in original scale
rmse_per_dim = np.sqrt(np.mean((Y_true_best - Y_pred_best) ** 2, axis=0))
std_per_dim  = np.std(Y_true_best, axis=0)
nrmse_per_dim = rmse_per_dim / std_per_dim

print("\nFinal best-model metrics in original scale:")
for d in range(n_dim):
    print(f"  {dim_names[d]}: RMSE={rmse_per_dim[d]:.6f}   "
          f"NRMSE={nrmse_per_dim[d]:.6f}")
print(f"  mean NRMSE = {np.mean(nrmse_per_dim):.6f}")


# ==========================================================
# VISUALIZATION: 3 PANELS, ONE PER COMPONENT
# ==========================================================
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

t_full = np.arange(test_len)
t_pred = np.arange(best_params["test_warmup"],
                   best_params["test_warmup"] + best_pred_len)

for d in range(n_dim):
    ax = axes[d]
    ax.plot(t_full, Y_test_raw[:, d],
            c="green", linewidth=1.0, label="Ground truth")
    ax.axvline(best_params["test_warmup"], linestyle="--",
               c="k", linewidth=1.0, label="Warmup end")
    ax.plot(t_pred, Y_pred_best[:, d], linestyle="--",
            c="red", linewidth=1.0, label="ESN closed-loop")
    ax.set_ylabel(dim_names[d])
    ax.set_title(f"{dim_names[d]}-component   "
                 f"NRMSE = {nrmse_per_dim[d]:.4f}")
    if d == 0:
        ax.legend(loc='upper right')

axes[-1].set_xlabel("Step within chosen test window")

fig.suptitle(
    f"3-D Lorenz Closed-Loop Prediction | mean NRMSE={np.mean(nrmse_per_dim):.4f} | "
    f"units={int(best_params['units'])}, "
    f"sr={best_params['sr']}, lr={best_params['lr']}, "
    f"in_sc={best_params['input_scaling']}, "
    f"reg={best_params['regression']:.0e}",
    y=1.00
)

plt.tight_layout()
plt.show()


# ==========================================================
# OPTIONAL: 3-D ATTRACTOR COMPARISON
# ==========================================================
# Useful sanity check: do the reconstructed dynamics live on the
# Lorenz butterfly, or has the RC drifted to a different attractor?
fig = plt.figure(figsize=(12, 5))

ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot(Y_true_best[:, 0], Y_true_best[:, 1], Y_true_best[:, 2],
         color='green', linewidth=0.5)
ax1.set_title('Ground truth attractor')
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')

ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.plot(Y_pred_best[:, 0], Y_pred_best[:, 1], Y_pred_best[:, 2],
         color='red', linewidth=0.5)
ax2.set_title('ESN closed-loop attractor')
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_zlabel('z')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# ==========================================================
# EXTRACT WEIGHTS FROM TRAINED reservoirpy ESN
# ==========================================================
# Pulls bare numpy arrays out of best_esn so the coupling simulations
# can run in plain numpy without the reservoirpy node machinery.
# ==========================================================

# Reservoir internal weights
W_res = best_reservoir.W
if hasattr(W_res, "toarray"):        # sparse -> dense
    W_res = W_res.toarray()
W_res = np.asarray(W_res)

# Input weights
W_in = best_reservoir.Win
if hasattr(W_in, "toarray"):
    W_in = W_in.toarray()
W_in = np.asarray(W_in)

# Reservoir bias (reservoirpy stores it as a column vector of shape (N, 1))
b_res = np.asarray(best_reservoir.bias).flatten()

# Readout weights and bias
# Ridge stores Wout of shape (N, n_out) and bias of shape (1, n_out) or (n_out,)
W_out = np.asarray(best_readout.Wout)
b_out = np.asarray(best_readout.bias).flatten()

# Leak rate and reservoir size
leak = float(best_params["lr"])
N    = int(best_params["units"])

print("Extracted ESN weights:")
print(f"  W_res : {W_res.shape}")
print(f"  W_in  : {W_in.shape}")
print(f"  b_res : {b_res.shape}")
print(f"  W_out : {W_out.shape}")
print(f"  b_out : {b_out.shape}")
print(f"  leak  : {leak}")
print(f"  N     : {N}")


# ==========================================================
# HELPER FUNCTIONS: single-step reservoir update + readout
# ==========================================================
# Standard leaky-integrator ESN update, matching reservoirpy's convention:
#   r(t+1) = (1 - leak) * r(t) + leak * tanh(W_res @ r(t) + W_in @ u(t) + b_res)
#   y(t)   = W_out.T @ r(t) + b_out
# ==========================================================
def reservoir_step(r, u, W_res, W_in, b_res, leak):
    """One leaky-integrator ESN step. r: (N,), u: (n_in,) -> r_new: (N,)."""
    pre = W_res @ r + W_in @ np.asarray(u).flatten() + b_res
    return (1.0 - leak) * r + leak * np.tanh(pre)


def readout(r, W_out, b_out):
    """Linear readout. r: (N,), returns y: (n_out,)."""
    return W_out.T @ r + b_out


# ==========================================================
# SANITY CHECK: reproduce a few steps of best_esn closed-loop
# ==========================================================
# Compares the numpy reimplementation against the reservoirpy ESN
# to make sure W_res / W_in / b_res / W_out / b_out / leak are
# consistent with the original model.
# ==========================================================
best_esn.reset()
_ = best_esn.run(X_test_best[:best_params["test_warmup"]])
# Snapshot reservoirpy's internal state right after warmup
_st = best_reservoir.state
if callable(_st):                       # v0.3: state is a method
    r_check = np.asarray(_st()).flatten().copy()
elif isinstance(_st, dict):             # v0.4: state is a dict with "out" key
    r_check = np.asarray(_st["out"]).flatten().copy()
else:                                    # fallback: already an array
    r_check = np.asarray(_st).flatten().copy()

# Run a few closed-loop steps inside reservoirpy
current_input = X_test_best[
    best_params["test_warmup"]:best_params["test_warmup"] + 1
]
y_rpy = []
for _ in range(5):
    pred = best_esn.run(current_input)
    y_rpy.append(np.asarray(pred).flatten())
    current_input = pred
y_rpy = np.array(y_rpy)

# Reproduce the same 5 steps in pure numpy starting from r_check
r = r_check.copy()
y_np = []
# First prediction uses r_check directly (matches reservoirpy timing)
y_prev = readout(r, W_out, b_out)
for _ in range(5):
    r = reservoir_step(r, y_prev, W_res, W_in, b_res, leak)
    y_prev = readout(r, W_out, b_out)
    y_np.append(y_prev.copy())
y_np = np.array(y_np)

err = np.max(np.abs(y_rpy - y_np))
print(f"\nSanity check: max abs diff between reservoirpy and numpy closed-loop = {err:.2e}")
if err < 1e-5:
    print("  ✓ numpy reimplementation matches reservoirpy")
else:
    print("  ✗ MISMATCH — check bias/transpose conventions before running coupling")

## Synchronization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# PREREQUISITES (from earlier cells)
# ==========================================================
# Assumes these are already in memory:
#   W_res, W_in, b_res, W_out, b_out  - extracted reservoir weights
#   leak                              - leak rate
#   N                                 - reservoir size
#   X_train_best                      - scaled training data (T, 3)
#   reservoir_step, readout           - helper functions

# ==========================================================
# ADJUSTABLE PARAMETERS — change these and re-run the cell
# ==========================================================
alpha           = 0.005      # coupling strength (0 = uncoupled), acts on reservoir state

starter_offset_1 = 10       # warm-up window start for RC 1
starter_offset_2 = 1000     # warm-up window start for RC 2

warmup_len      = 500       # warm-up duration (steps)
n_transient     = 4000      # uncoupled closed-loop steps to land on the RC's attractor
n_steps         = 4000      # length of coupled simulation to plot
plot_steps      = 4000      # how many steps of the coupled result to display

# ==========================================================
# WARM UP TWO RESERVOIRS WITH DIFFERENT SEGMENTS
# ==========================================================
def warmup_reservoir(driving_signal, W_res, W_in, b_res, leak, N):
    r = np.zeros(N)
    for u in driving_signal:
        r = reservoir_step(r, u, W_res, W_in, b_res, leak)
    return r

# Validate offsets fit in the warm-up source
assert starter_offset_1 + warmup_len <= len(X_train_best), \
    f"starter_offset_1 + warmup_len exceeds data length {len(X_train_best)}"
assert starter_offset_2 + warmup_len <= len(X_train_best), \
    f"starter_offset_2 + warmup_len exceeds data length {len(X_train_best)}"

seg1 = X_train_best[starter_offset_1:starter_offset_1 + warmup_len]
seg2 = X_train_best[starter_offset_2:starter_offset_2 + warmup_len]

r1_0 = warmup_reservoir(seg1, W_res, W_in, b_res, leak, N)
r2_0 = warmup_reservoir(seg2, W_res, W_in, b_res, leak, N)

# ==========================================================
# UNCOUPLED CLOSED-LOOP TRANSIENT
# ==========================================================
# Each RC runs autonomously (alpha = 0) for n_transient steps to settle
# onto its own chaotic attractor. Ensures both reservoirs start the
# coupled simulation from genuinely on-attractor states.
def closed_loop_uncoupled(r0, n_steps, W_res, W_in, b_res, W_out, b_out, leak):
    r = r0.copy()
    for _ in range(n_steps):
        y = readout(r, W_out, b_out)
        r = reservoir_step(r, y, W_res, W_in, b_res, leak)
        if not np.all(np.isfinite(y)):
            return None
    return r

r1_on = closed_loop_uncoupled(r1_0, n_transient,
                              W_res, W_in, b_res, W_out, b_out, leak)
r2_on = closed_loop_uncoupled(r2_0, n_transient + 200,    # +200 offset for variety
                              W_res, W_in, b_res, W_out, b_out, leak)

assert r1_on is not None and r2_on is not None, \
    "RC diverged during uncoupled transient — check training quality."

y1_init = readout(r1_on, W_out, b_out)
y2_init = readout(r2_on, W_out, b_out)

print(f"Warm-up + uncoupled transient complete:")
print(f"  RC 1 from offset {starter_offset_1}: "
      f"y_init = [{y1_init[0]:+.3f} {y1_init[1]:+.3f} {y1_init[2]:+.3f}]")
print(f"  RC 2 from offset {starter_offset_2}: "
      f"y_init = [{y2_init[0]:+.3f} {y2_init[1]:+.3f} {y2_init[2]:+.3f}]")
print(f"  ||r1_on - r2_on|| = {np.linalg.norm(r1_on - r2_on):.3f}")

# ==========================================================
# COUPLED CLOSED-LOOP SIMULATION — STATE-LEVEL DIFFUSIVE COUPLING
# ==========================================================
#
# Option (b): coupling acts directly on the hidden state, not on the input.
#
#   r1_pre = F(r1, y1)         # standard ESN update with own prediction
#   r2_pre = F(r2, y2)
#   r1_new = r1_pre + alpha * (r2_pre - r1_pre)     # diffusive correction
#   r2_new = r2_pre + alpha * (r1_pre - r2_pre)
#
# The synchronization manifold {r1 = r2} is invariant: when r1 = r2 the
# coupling term vanishes identically, so the synchronized dynamics are
# exactly those of a single uncoupled closed-loop ESN. alpha controls
# only the transverse stability of this manifold.
#
# Note: we use the *pre-coupling* states r{1,2}_pre on the right-hand side
# for both, so the update is symmetric and corresponds to the standard
# diffusive form e(t+1) ~ [J - 2*alpha*I] e(t) for the error variable.
# ==========================================================
r1 = r1_on.copy()
r2 = r2_on.copy()

y1_traj = np.zeros((n_steps, 3))
y2_traj = np.zeros((n_steps, 3))

diverged = False
for t in range(n_steps):
    # Each reservoir reads out its OWN state — no input mixing
    y1 = readout(r1, W_out, b_out)
    y2 = readout(r2, W_out, b_out)

    # Each reservoir advances with its OWN prediction as input
    r1_pre = reservoir_step(r1, y1, W_res, W_in, b_res, leak)
    r2_pre = reservoir_step(r2, y2, W_res, W_in, b_res, leak)

    # Diffusive coupling applied AFTER the state update — option (b)
    r1 = r1_pre + alpha * (r2_pre - r1_pre)
    r2 = r2_pre + alpha * (r1_pre - r2_pre)

    y1_traj[t] = y1
    y2_traj[t] = y2

    if not (np.all(np.isfinite(r1)) and np.all(np.isfinite(r2))):
        print(f"  ✗ diverged at step {t}")
        y1_traj = y1_traj[:t]
        y2_traj = y2_traj[:t]
        diverged = True
        break

# ==========================================================
# DIAGNOSTICS
# ==========================================================
if not diverged:
    diff = np.linalg.norm(y1_traj - y2_traj, axis=1)
    E_full   = float(np.mean(diff))
    E_late   = float(np.mean(diff[len(diff) // 2:]))
    print(f"\nSimulation complete: {n_steps} steps, alpha = {alpha}")
    print(f"  E (full window)        = {E_full:.5f}")
    print(f"  E (second half only)   = {E_late:.5f}")
    print(f"  ||y1 - y2|| at start   = {diff[0]:.5f}")
    print(f"  ||y1 - y2|| at end     = {diff[-1]:.5f}")
    print(f"  Std of y1[:,0] late    = {y1_traj[len(y1_traj)//2:, 0].std():.4f}")
    print(f"  Std of y2[:,0] late    = {y2_traj[len(y2_traj)//2:, 0].std():.4f}")

# ==========================================================
# PLOT: 3 COMPONENTS OVERLAID + DISTANCE OVER TIME
# ==========================================================
n_plot = min(plot_steps, len(y1_traj))
t_axis = np.arange(n_plot)
dim_names = ['x', 'y', 'z']

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)

for d in range(3):
    ax = axes[d]
    ax.plot(t_axis, y1_traj[:n_plot, d], color='steelblue',
            linewidth=0.8, label=f'RC 1 (offset {starter_offset_1})')
    ax.plot(t_axis, y2_traj[:n_plot, d], color='crimson',
            linewidth=0.8, alpha=0.8,
            label=f'RC 2 (offset {starter_offset_2})')
    ax.set_ylabel(f'{dim_names[d]} (scaled)')
    if d == 0:
        ax.legend(loc='upper right', fontsize=9)

if not diverged:
    diff = np.linalg.norm(y1_traj - y2_traj, axis=1)
    axes[3].plot(np.arange(len(diff))[:n_plot], diff[:n_plot],
                 color='black', linewidth=0.8)
    axes[3].set_ylabel(r'$\|y_1 - y_2\|$')
    axes[3].axhline(0, color='gray', linestyle=':', linewidth=0.5)
    axes[3].grid(alpha=0.3)
else:
    axes[3].text(0.5, 0.5, 'diverged', ha='center', va='center',
                 transform=axes[3].transAxes)

axes[-1].set_xlabel('Step (after uncoupled transient)')

fig.suptitle(
    f'Two coupled identical RCs — STATE-LEVEL coupling   |   '
    fr'$\alpha$ = {alpha}   |   '
    f'starters = ({starter_offset_1}, {starter_offset_2})   |   '
    + (f'E = {E_late:.4f}' if not diverged else 'DIVERGED'),
    y=1.00
)
plt.tight_layout()
#plt.savefig("lorenz_ESN_state_coupling.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# PREREQUISITES (from earlier cells)
# ==========================================================
# Assumes these are already in memory:
#   W_res, W_in, b_res, W_out, b_out  - extracted reservoir weights
#   leak                              - leak rate
#   N                                 - reservoir size
#   X_train_best                      - scaled training data (T, 3)
#   reservoir_step, readout           - helper functions
#
# (All produced by the weight-extraction cell.)

# ==========================================================
# ADJUSTABLE PARAMETERS — change these and re-run the cell
# ==========================================================
# --- coupling channel ---
coupling_dim    = 3        # d: dimensionality of the communication channel
                            #    d = N recovers full identity-style coupling
                            #    d = 1 is near rank-1 (readout-like) coupling
alpha           = 0.05      # coupling strength per step (0 = uncoupled)

# --- initial conditions ---
starter_offset_1 = 10       # warm-up window start for RC 1
starter_offset_2 = 1000     # warm-up window start for RC 2

# --- simulation lengths ---
warmup_len      = 500       # warm-up duration (steps)
n_transient     = 4000      # uncoupled closed-loop steps to land on the RC's attractor
n_steps         = 4000      # length of coupled simulation
plot_steps      = 4000      # how many steps of the coupled result to display

# --- channel construction ---
pc_traj_len     = 6000      # length of uncoupled trajectory used to estimate
                            #   the principal-component subspace for C
pc_traj_discard = 1000      # discard this many initial steps of that trajectory

# --- sweeps (set to None to skip) ---
alpha_sweep     = np.linspace(0.0, 0.20, 41)          # alpha grid at fixed coupling_dim
dim_sweep       = [1, 3, 10, 30, 100, 300, N]         # d grid for alpha_c(d) curve
dim_sweep_alpha = np.linspace(0.0, 0.40, 41)          # alpha grid used inside dim_sweep
sync_threshold  = 1e-3      # E_late below this counts as "synchronized"


# ==========================================================
# WARM UP TWO RESERVOIRS WITH DIFFERENT SEGMENTS
# ==========================================================
def warmup_reservoir(driving_signal, W_res, W_in, b_res, leak, N):
    r = np.zeros(N)
    for u in driving_signal:
        r = reservoir_step(r, u, W_res, W_in, b_res, leak)
    return r

assert starter_offset_1 + warmup_len <= len(X_train_best), \
    f"starter_offset_1 + warmup_len exceeds data length {len(X_train_best)}"
assert starter_offset_2 + warmup_len <= len(X_train_best), \
    f"starter_offset_2 + warmup_len exceeds data length {len(X_train_best)}"

seg1 = X_train_best[starter_offset_1:starter_offset_1 + warmup_len]
seg2 = X_train_best[starter_offset_2:starter_offset_2 + warmup_len]

r1_0 = warmup_reservoir(seg1, W_res, W_in, b_res, leak, N)
r2_0 = warmup_reservoir(seg2, W_res, W_in, b_res, leak, N)


# ==========================================================
# UNCOUPLED CLOSED-LOOP TRANSIENT
# ==========================================================
# Each RC runs autonomously (alpha = 0) to settle onto its own chaotic
# attractor before coupling is switched on. Also used to collect the
# state trajectory from which the coupling subspace is estimated.
def closed_loop_uncoupled(r0, n_steps, W_res, W_in, b_res, W_out, b_out, leak,
                          record=False):
    r = r0.copy()
    states = np.zeros((n_steps, len(r))) if record else None
    for k in range(n_steps):
        y = readout(r, W_out, b_out)
        r = reservoir_step(r, y, W_res, W_in, b_res, leak)
        if not np.all(np.isfinite(y)):
            return (None, None) if record else None
        if record:
            states[k] = r
    return (r, states) if record else r

# RC 1: settle on attractor AND record states for the PC subspace
r1_on, R_traj = closed_loop_uncoupled(
    r1_0, max(n_transient, pc_traj_len),
    W_res, W_in, b_res, W_out, b_out, leak, record=True)

# RC 2: settle on attractor (slightly longer run for variety)
r2_on = closed_loop_uncoupled(
    r2_0, n_transient + 200,
    W_res, W_in, b_res, W_out, b_out, leak)

assert r1_on is not None and r2_on is not None, \
    "RC diverged during uncoupled transient — check training quality."

y1_init = readout(r1_on, W_out, b_out)
y2_init = readout(r2_on, W_out, b_out)

print("Warm-up + uncoupled transient complete:")
print(f"  RC 1 from offset {starter_offset_1}: "
      f"y_init = [{y1_init[0]:+.3f} {y1_init[1]:+.3f} {y1_init[2]:+.3f}]")
print(f"  RC 2 from offset {starter_offset_2}: "
      f"y_init = [{y2_init[0]:+.3f} {y2_init[1]:+.3f} {y2_init[2]:+.3f}]")
print(f"  ||r1_on - r2_on|| = {np.linalg.norm(r1_on - r2_on):.3f}")


# ==========================================================
# BUILD THE COUPLING SUBSPACE  C = V_d V_d^T
# ==========================================================
# V_d holds the top-d principal components of the reservoir's autonomous
# chaotic activity — the directions in state space where the dynamics
# carry the most variance ("macroscopic modes"). Coupling through
# C = V_d V_d^T means the two reservoirs exchange information only through
# this d-dimensional communication channel; the remaining low-variance
# directions evolve independently.
#
# Properties:
#   - basis-free: V_d is fixed by the dynamics, not by neuron labelling
#   - d = N  -> C = I  (full identity coupling, the dense baseline)
#   - d = 1  -> rank-1 coupling, close to readout-only coupling
#   - the synchronization manifold {r1 = r2} stays invariant for any C,
#     since C(r2 - r1) = 0 there regardless of C.
# ==========================================================
def build_coupling_matrix(R_traj, d, discard):
    """Projector onto the top-d principal-component subspace of R_traj."""
    R = R_traj[discard:]
    R = R - R.mean(axis=0, keepdims=True)
    # economy SVD: rows of Vt are principal directions in state space
    _, S, Vt = np.linalg.svd(R, full_matrices=False)
    d = int(min(d, Vt.shape[0]))
    V_d = Vt[:d].T                       # (N, d)
    C = V_d @ V_d.T                      # (N, N) symmetric projector
    var_ratio = float(np.sum(S[:d] ** 2) / np.sum(S ** 2))
    return C, V_d, S, var_ratio

C, V_d, S_spec, var_captured = build_coupling_matrix(
    R_traj, coupling_dim, pc_traj_discard)

print(f"\nCoupling channel C = V_d V_d^T")
print(f"  requested d              = {coupling_dim}")
print(f"  effective d              = {V_d.shape[1]}")
print(f"  variance captured by C   = {var_captured:.4f}")
print(f"  trace(C) (= rank)        = {np.trace(C):.1f}")


# ==========================================================
# COUPLED CLOSED-LOOP SIMULATION — MATRIX COUPLING
# ==========================================================
# Option (b) with a coupling matrix:
#   r1_pre = F(r1, y1)
#   r2_pre = F(r2, y2)
#   r1 = r1_pre + alpha * C @ (r2_pre - r1_pre)
#   r2 = r2_pre + alpha * C @ (r1_pre - r2_pre)
#
# Each reservoir reads out its OWN state and advances with its OWN
# prediction; the only inter-reservoir term is the diffusive correction
# projected through C. Pre-coupling states r{1,2}_pre are used on both
# right-hand sides so the update is symmetric and the error variable
# obeys  e(t+1) ~ [J - 2*alpha*C] e(t)  near the sync manifold.
# ==========================================================
def run_coupled(r1_start, r2_start, C, alpha, n_steps,
                W_res, W_in, b_res, W_out, b_out, leak,
                record=True):
    r1 = r1_start.copy()
    r2 = r2_start.copy()
    y1_traj = np.zeros((n_steps, 3)) if record else None
    y2_traj = np.zeros((n_steps, 3)) if record else None

    for t in range(n_steps):
        y1 = readout(r1, W_out, b_out)
        y2 = readout(r2, W_out, b_out)

        r1_pre = reservoir_step(r1, y1, W_res, W_in, b_res, leak)
        r2_pre = reservoir_step(r2, y2, W_res, W_in, b_res, leak)

        delta = r2_pre - r1_pre
        cpl   = alpha * (C @ delta)
        r1 = r1_pre + cpl
        r2 = r2_pre - cpl                      # C @ (r1_pre - r2_pre) = -cpl

        if record:
            y1_traj[t] = y1
            y2_traj[t] = y2

        if not (np.all(np.isfinite(r1)) and np.all(np.isfinite(r2))):
            if record:
                return y1_traj[:t], y2_traj[:t], True
            return None, None, True

    if record:
        return y1_traj, y2_traj, False
    return r1, r2, False


y1_traj, y2_traj, diverged = run_coupled(
    r1_on, r2_on, C, alpha, n_steps,
    W_res, W_in, b_res, W_out, b_out, leak, record=True)


# ==========================================================
# DIAGNOSTICS
# ==========================================================
if not diverged:
    diff = np.linalg.norm(y1_traj - y2_traj, axis=1)
    E_full = float(np.mean(diff))
    E_late = float(np.mean(diff[len(diff) // 2:]))
    print(f"\nSimulation complete: {n_steps} steps, "
          f"alpha = {alpha}, d = {V_d.shape[1]}")
    print(f"  E (full window)        = {E_full:.5f}")
    print(f"  E (second half only)   = {E_late:.5f}")
    print(f"  ||y1 - y2|| at start   = {diff[0]:.5f}")
    print(f"  ||y1 - y2|| at end     = {diff[-1]:.5f}")
    print(f"  synchronized?          = {E_late < sync_threshold}")
else:
    E_late = np.nan
    print(f"\n  ✗ diverged — alpha = {alpha}, d = {V_d.shape[1]}")


# ==========================================================
# PLOT 1: 3 COMPONENTS OVERLAID + DISTANCE OVER TIME
# ==========================================================
n_plot = min(plot_steps, len(y1_traj))
t_axis = np.arange(n_plot)
dim_names = ['x', 'y', 'z']

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
for d_ in range(3):
    ax = axes[d_]
    ax.plot(t_axis, y1_traj[:n_plot, d_], color='steelblue',
            linewidth=0.8, label=f'RC 1 (offset {starter_offset_1})')
    ax.plot(t_axis, y2_traj[:n_plot, d_], color='crimson',
            linewidth=0.8, alpha=0.8, label=f'RC 2 (offset {starter_offset_2})')
    ax.set_ylabel(f'{dim_names[d_]} (scaled)')
    if d_ == 0:
        ax.legend(loc='upper right', fontsize=9)

if not diverged:
    diff = np.linalg.norm(y1_traj - y2_traj, axis=1)
    axes[3].plot(np.arange(len(diff))[:n_plot], diff[:n_plot],
                 color='black', linewidth=0.8)
    axes[3].set_ylabel(r'$\|y_1 - y_2\|$')
    axes[3].axhline(0, color='gray', linestyle=':', linewidth=0.5)
    axes[3].grid(alpha=0.3)
else:
    axes[3].text(0.5, 0.5, 'diverged', ha='center', va='center',
                 transform=axes[3].transAxes)

axes[-1].set_xlabel('Step (after uncoupled transient)')
fig.suptitle(
    f'Matrix-coupled identical RCs   |   '
    fr'$\alpha$ = {alpha}   |   d = {V_d.shape[1]}   |   '
    f'var(C) = {var_captured:.3f}   |   '
    + (f'E = {E_late:.4f}' if not diverged else 'DIVERGED'),
    y=1.00)
plt.tight_layout()
plt.savefig("lorenz_ESN_matrix_coupling.png", dpi=120)
plt.show()


# ==========================================================
# SWEEP 1:  E_late vs alpha  at fixed coupling_dim
# ==========================================================
if alpha_sweep is not None:
    print(f"\nSweeping alpha at d = {V_d.shape[1]} "
          f"({len(alpha_sweep)} values)...")
    E_vs_alpha = np.full(len(alpha_sweep), np.nan)
    for i, a in enumerate(alpha_sweep):
        r1f, r2f, div = run_coupled(
            r1_on, r2_on, C, a, n_steps,
            W_res, W_in, b_res, W_out, b_out, leak, record=True)
        if not div:
            dd = np.linalg.norm(r1f - r2f, axis=1)
            E_vs_alpha[i] = np.mean(dd[len(dd) // 2:])

    # estimate threshold: first alpha where E_late drops below sync_threshold
    below = np.where(E_vs_alpha < sync_threshold)[0]
    alpha_c = alpha_sweep[below[0]] if len(below) else np.nan
    print(f"  estimated alpha_c (d = {V_d.shape[1]}) = {alpha_c:.4f}")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.semilogy(alpha_sweep, E_vs_alpha, 'o-', color='darkblue',
                markersize=4, linewidth=1)
    ax.axhline(sync_threshold, color='gray', linestyle='--', linewidth=1,
               label=f'sync threshold = {sync_threshold:g}')
    if np.isfinite(alpha_c):
        ax.axvline(alpha_c, color='crimson', linestyle=':', linewidth=1.5,
                   label=fr'$\alpha_c \approx$ {alpha_c:.3f}')
    ax.set_xlabel(r'coupling strength $\alpha$')
    ax.set_ylabel(r'late-window sync error  $E_{\rm late}$')
    ax.set_title(f'Synchronization transition  (d = {V_d.shape[1]}, '
                 f'channel variance = {var_captured:.3f})')
    ax.legend()
    ax.grid(alpha=0.3, which='both')
    plt.tight_layout()
    plt.savefig("lorenz_ESN_alpha_sweep.png", dpi=120)
    plt.show()


# ==========================================================
# SWEEP 2:  alpha_c  vs  coupling channel dimension d
# ==========================================================
# The main scientific output: how does the synchronization threshold
# depend on the bandwidth of the communication channel? Expect alpha_c
# to grow as d shrinks (narrower channel -> stronger per-step coupling
# needed), and possibly to diverge below some critical dimension d*.
# ==========================================================
if dim_sweep is not None:
    print(f"\nSweeping coupling dimension d over {dim_sweep} ...")
    alpha_c_vs_d = np.full(len(dim_sweep), np.nan)
    var_vs_d     = np.full(len(dim_sweep), np.nan)

    for j, d_val in enumerate(dim_sweep):
        C_d, V_dd, _, var_d = build_coupling_matrix(
            R_traj, d_val, pc_traj_discard)
        var_vs_d[j] = var_d

        E_line = np.full(len(dim_sweep_alpha), np.nan)
        for i, a in enumerate(dim_sweep_alpha):
            r1f, r2f, div = run_coupled(
                r1_on, r2_on, C_d, a, n_steps,
                W_res, W_in, b_res, W_out, b_out, leak, record=True)
            if not div:
                dd = np.linalg.norm(r1f - r2f, axis=1)
                E_line[i] = np.mean(dd[len(dd) // 2:])

        below = np.where(E_line < sync_threshold)[0]
        alpha_c_vs_d[j] = dim_sweep_alpha[below[0]] if len(below) else np.nan
        tag = (f"{alpha_c_vs_d[j]:.4f}" if np.isfinite(alpha_c_vs_d[j])
               else "no sync in range")
        print(f"  d = {V_dd.shape[1]:4d}  "
              f"var(C) = {var_d:.4f}   alpha_c = {tag}")

    fig, ax = plt.subplots(1, 2, figsize=(13, 5))

    ax[0].plot(dim_sweep, alpha_c_vs_d, 'o-', color='darkgreen',
               markersize=6, linewidth=1.2)
    ax[0].set_xlabel('coupling channel dimension  d')
    ax[0].set_ylabel(r'synchronization threshold  $\alpha_c$')
    ax[0].set_title(r'$\alpha_c(d)$  —  narrower channel needs stronger coupling')
    ax[0].set_xscale('log')
    ax[0].grid(alpha=0.3, which='both')

    ax[1].plot(var_vs_d, alpha_c_vs_d, 's-', color='purple',
               markersize=6, linewidth=1.2)
    ax[1].set_xlabel('variance captured by channel  C')
    ax[1].set_ylabel(r'synchronization threshold  $\alpha_c$')
    ax[1].set_title(r'$\alpha_c$ vs channel variance')
    ax[1].grid(alpha=0.3)

    fig.suptitle('Synchronization threshold vs communication-channel bandwidth',
                 y=1.02)
    plt.tight_layout()
    #plt.savefig("lorenz_ESN_alphac_vs_dim.png", dpi=120)
    plt.show()

    print("\nSummary table:")
    print(f"  {'d':>6}  {'var(C)':>9}  {'alpha_c':>10}")
    for d_val, vr, ac in zip(dim_sweep, var_vs_d, alpha_c_vs_d):
        ac_s = f"{ac:.4f}" if np.isfinite(ac) else "   --   "
        print(f"  {d_val:>6}  {vr:>9.4f}  {ac_s:>10}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp

# ==========================================================
# PREREQUISITES (from earlier cells)
# ==========================================================
# Assumes these are already in memory:
#   W_res, W_in, b_res, W_out, b_out  - extracted reservoir weights
#   leak                              - leak rate
#   N                                 - reservoir size
#   X_train_best                      - scaled training data (T, 3)
#   reservoir_step, readout           - helper functions
#
# NOTE: the standard reservoir_step / readout helpers are still used for
# warm-up and the uncoupled transient. The coupled loop below uses its
# OWN inline update because the coupling term goes INSIDE the tanh and
# cannot be expressed through the plain reservoir_step helper.

# ==========================================================
# ADJUSTABLE PARAMETERS — change these and re-run the cell
# ==========================================================
# --- coupling channel: random sparse C ---
coupling_density = 0.01     # fraction of nonzero entries in C (per row ~ density*N)
coupling_scale   = 1.0      # magnitude scale of nonzero entries of C
coupling_seed    = 7        # RNG seed for the sparse coupling pattern
symmetric_C      = True     # if True, symmetrise C (C <- (C + C^T)/2)
alpha            = 0.05     # coupling strength (0 = uncoupled)

# --- initial conditions ---
starter_offset_1 = 10       # warm-up window start for RC 1
starter_offset_2 = 1000     # warm-up window start for RC 2

# --- simulation lengths ---
warmup_len      = 500       # warm-up duration (steps)
n_transient     = 4000      # uncoupled closed-loop steps to land on the attractor
n_steps         = 4000      # length of coupled simulation
plot_steps      = 4000      # how many steps of the coupled result to display

# --- sweeps (set to None to skip) ---
alpha_sweep      = np.linspace(0.0, 0.40, 41)   # alpha grid at fixed density
density_sweep    = [0.005, 0.01, 0.02, 0.05, 0.1, 0.3, 1.0]
density_alpha    = np.linspace(0.0, 0.80, 41)   # alpha grid used inside density sweep
sync_threshold   = 1e-3     # E_late below this counts as "synchronized"


# ==========================================================
# BUILD RANDOM SPARSE COUPLING MATRIX C
# ==========================================================
# Each entry is nonzero with probability `coupling_density`; nonzero
# values are uniform in [-coupling_scale/2, +coupling_scale/2]. Neuron i
# in one reservoir is coupled to a random handful of neurons in the
# other — a narrow, unstructured communication channel.
#
# C(r_B - r_A) is still zero on the sync manifold r_A = r_B, so the
# manifold stays invariant and the synchronized dynamics are unchanged.
# ==========================================================
def build_sparse_coupling(N, density, scale, seed, symmetric):
    rng = np.random.default_rng(seed)
    mask = rng.random((N, N)) < density
    vals = (rng.random((N, N)) - 0.5) * scale
    C = np.where(mask, vals, 0.0)
    if symmetric:
        C = 0.5 * (C + C.T)
    return C

C = build_sparse_coupling(N, coupling_density, coupling_scale,
                          coupling_seed, symmetric_C)
nnz_frac = np.count_nonzero(C) / C.size
print(f"Random sparse coupling matrix C:")
print(f"  shape            = {C.shape}")
print(f"  density (actual) = {nnz_frac:.4f}")
print(f"  ||C||_2          = {np.linalg.norm(C, 2):.4f}")
print(f"  symmetric        = {symmetric_C}")


# ==========================================================
# WARM UP TWO RESERVOIRS WITH DIFFERENT SEGMENTS
# ==========================================================
def warmup_reservoir(driving_signal, W_res, W_in, b_res, leak, N):
    r = np.zeros(N)
    for u in driving_signal:
        r = reservoir_step(r, u, W_res, W_in, b_res, leak)
    return r

assert starter_offset_1 + warmup_len <= len(X_train_best), \
    f"starter_offset_1 + warmup_len exceeds data length {len(X_train_best)}"
assert starter_offset_2 + warmup_len <= len(X_train_best), \
    f"starter_offset_2 + warmup_len exceeds data length {len(X_train_best)}"

seg1 = X_train_best[starter_offset_1:starter_offset_1 + warmup_len]
seg2 = X_train_best[starter_offset_2:starter_offset_2 + warmup_len]

r1_0 = warmup_reservoir(seg1, W_res, W_in, b_res, leak, N)
r2_0 = warmup_reservoir(seg2, W_res, W_in, b_res, leak, N)


# ==========================================================
# UNCOUPLED CLOSED-LOOP TRANSIENT
# ==========================================================
def closed_loop_uncoupled(r0, n_steps, W_res, W_in, b_res, W_out, b_out, leak):
    r = r0.copy()
    for _ in range(n_steps):
        y = readout(r, W_out, b_out)
        r = reservoir_step(r, y, W_res, W_in, b_res, leak)
        if not np.all(np.isfinite(y)):
            return None
    return r

r1_on = closed_loop_uncoupled(r1_0, n_transient,
                              W_res, W_in, b_res, W_out, b_out, leak)
r2_on = closed_loop_uncoupled(r2_0, n_transient + 200,
                              W_res, W_in, b_res, W_out, b_out, leak)

assert r1_on is not None and r2_on is not None, \
    "RC diverged during uncoupled transient — check training quality."

y1_init = readout(r1_on, W_out, b_out)
y2_init = readout(r2_on, W_out, b_out)
print(f"\nWarm-up + uncoupled transient complete:")
print(f"  RC 1 from offset {starter_offset_1}: "
      f"y_init = [{y1_init[0]:+.3f} {y1_init[1]:+.3f} {y1_init[2]:+.3f}]")
print(f"  RC 2 from offset {starter_offset_2}: "
      f"y_init = [{y2_init[0]:+.3f} {y2_init[1]:+.3f} {y2_init[2]:+.3f}]")
print(f"  ||r1_on - r2_on|| = {np.linalg.norm(r1_on - r2_on):.3f}")


# ==========================================================
# COUPLED CLOSED-LOOP SIMULATION — COUPLING INSIDE THE TANH
# ==========================================================
# Update rule (coupling term sits INSIDE the activation):
#
#   pre_A = W_res @ r_A + W_in @ y_A + b_res + alpha * C @ (r_B - r_A)
#   pre_B = W_res @ r_B + W_in @ y_B + b_res + alpha * C @ (r_A - r_B)
#   r_A <- (1 - leak) * r_A + leak * tanh(pre_A)
#   r_B <- (1 - leak) * r_B + leak * tanh(pre_B)
#
# Both reservoirs read out their OWN state and feed their OWN prediction
# back as input; the only inter-reservoir term is alpha * C * (state
# difference) added to the pre-activation. Because it acts inside tanh,
# the effective coupling each neuron feels is modulated by the local
# activation slope (1 - tanh^2): saturated neurons barely couple,
# near-linear neurons couple fully.
#
# The sync manifold r_A = r_B is still invariant: there C(r_B - r_A) = 0
# and the term drops out, so synchronized dynamics = single uncoupled ESN.
# ==========================================================
def run_coupled_inside(r1_start, r2_start, C, alpha, n_steps,
                       W_res, W_in, b_res, W_out, b_out, leak,
                       record=True):
    r1 = r1_start.copy()
    r2 = r2_start.copy()
    y1_traj = np.zeros((n_steps, 3)) if record else None
    y2_traj = np.zeros((n_steps, 3)) if record else None

    for t in range(n_steps):
        y1 = readout(r1, W_out, b_out)
        y2 = readout(r2, W_out, b_out)

        delta = r2 - r1                       # state difference
        cpl   = alpha * (C @ delta)           # C @ (r1 - r2) = -cpl

        pre1 = W_res @ r1 + W_in @ y1 + b_res + cpl
        pre2 = W_res @ r2 + W_in @ y2 + b_res - cpl

        r1 = (1.0 - leak) * r1 + leak * np.tanh(pre1)
        r2 = (1.0 - leak) * r2 + leak * np.tanh(pre2)

        if record:
            y1_traj[t] = y1
            y2_traj[t] = y2

        if not (np.all(np.isfinite(r1)) and np.all(np.isfinite(r2))):
            if record:
                return y1_traj[:t], y2_traj[:t], True
            return None, None, True

    if record:
        return y1_traj, y2_traj, False
    return r1, r2, False


y1_traj, y2_traj, diverged = run_coupled_inside(
    r1_on, r2_on, C, alpha, n_steps,
    W_res, W_in, b_res, W_out, b_out, leak, record=True)


# ==========================================================
# DIAGNOSTICS
# ==========================================================
if not diverged:
    diff = np.linalg.norm(y1_traj - y2_traj, axis=1)
    E_full = float(np.mean(diff))
    E_late = float(np.mean(diff[len(diff) // 2:]))
    print(f"\nSimulation complete: {n_steps} steps, "
          f"alpha = {alpha}, density = {nnz_frac:.4f}")
    print(f"  E (full window)        = {E_full:.5f}")
    print(f"  E (second half only)   = {E_late:.5f}")
    print(f"  ||y1 - y2|| at start   = {diff[0]:.5f}")
    print(f"  ||y1 - y2|| at end     = {diff[-1]:.5f}")
    print(f"  synchronized?          = {E_late < sync_threshold}")
else:
    E_late = np.nan
    print(f"\n  ✗ diverged — alpha = {alpha}, density = {nnz_frac:.4f}")


# ==========================================================
# PLOT 1: 3 COMPONENTS OVERLAID + DISTANCE OVER TIME
# ==========================================================
n_plot = min(plot_steps, len(y1_traj))
t_axis = np.arange(n_plot)
dim_names = ['x', 'y', 'z']

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
for d_ in range(3):
    ax = axes[d_]
    ax.plot(t_axis, y1_traj[:n_plot, d_], color='steelblue',
            linewidth=0.8, label=f'RC 1 (offset {starter_offset_1})')
    ax.plot(t_axis, y2_traj[:n_plot, d_], color='crimson',
            linewidth=0.8, alpha=0.8, label=f'RC 2 (offset {starter_offset_2})')
    ax.set_ylabel(f'{dim_names[d_]} (scaled)')
    if d_ == 0:
        ax.legend(loc='upper right', fontsize=9)

if not diverged:
    diff = np.linalg.norm(y1_traj - y2_traj, axis=1)
    axes[3].plot(np.arange(len(diff))[:n_plot], diff[:n_plot],
                 color='black', linewidth=0.8)
    axes[3].set_ylabel(r'$\|y_1 - y_2\|$')
    axes[3].axhline(0, color='gray', linestyle=':', linewidth=0.5)
    axes[3].grid(alpha=0.3)
else:
    axes[3].text(0.5, 0.5, 'diverged', ha='center', va='center',
                 transform=axes[3].transAxes)

axes[-1].set_xlabel('Step (after uncoupled transient)')
fig.suptitle(
    f'Sparse coupling INSIDE tanh   |   '
    fr'$\alpha$ = {alpha}   |   density = {nnz_frac:.3f}   |   '
    + (f'E = {E_late:.4f}' if not diverged else 'DIVERGED'),
    y=1.00)
plt.tight_layout()
plt.savefig("lorenz_ESN_sparse_inside_coupling.png", dpi=120)
plt.show()


# ==========================================================
# SWEEP 1:  E_late vs alpha  at fixed coupling density
# ==========================================================
if alpha_sweep is not None:
    print(f"\nSweeping alpha at density = {nnz_frac:.4f} "
          f"({len(alpha_sweep)} values)...")
    E_vs_alpha = np.full(len(alpha_sweep), np.nan)
    for i, a in enumerate(alpha_sweep):
        r1f, r2f, div = run_coupled_inside(
            r1_on, r2_on, C, a, n_steps,
            W_res, W_in, b_res, W_out, b_out, leak, record=True)
        if not div:
            dd = np.linalg.norm(r1f - r2f, axis=1)
            E_vs_alpha[i] = np.mean(dd[len(dd) // 2:])

    below = np.where(E_vs_alpha < sync_threshold)[0]
    alpha_c = alpha_sweep[below[0]] if len(below) else np.nan
    print(f"  estimated alpha_c (density = {nnz_frac:.4f}) = {alpha_c:.4f}")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.semilogy(alpha_sweep, E_vs_alpha, 'o-', color='darkblue',
                markersize=4, linewidth=1)
    ax.axhline(sync_threshold, color='gray', linestyle='--', linewidth=1,
               label=f'sync threshold = {sync_threshold:g}')
    if np.isfinite(alpha_c):
        ax.axvline(alpha_c, color='crimson', linestyle=':', linewidth=1.5,
                   label=fr'$\alpha_c \approx$ {alpha_c:.3f}')
    ax.set_xlabel(r'coupling strength $\alpha$')
    ax.set_ylabel(r'late-window sync error  $E_{\rm late}$')
    ax.set_title(f'Synchronization transition  '
                 f'(sparse C inside tanh, density = {nnz_frac:.3f})')
    ax.legend()
    ax.grid(alpha=0.3, which='both')
    plt.tight_layout()
    plt.savefig("lorenz_ESN_sparse_inside_alpha_sweep.png", dpi=120)
    plt.show()


# ==========================================================
# SWEEP 2:  alpha_c  vs  coupling density
# ==========================================================
# How does the synchronization threshold depend on how many neuron-to-
# neuron links the channel has? Expect alpha_c to grow as density falls
# (fewer links -> stronger per-link coupling needed), with the inside-
# tanh modulation pushing thresholds higher than the outside-tanh case.
# ==========================================================
if density_sweep is not None:
    print(f"\nSweeping coupling density over {density_sweep} ...")
    alpha_c_vs_dens = np.full(len(density_sweep), np.nan)
    actual_dens     = np.full(len(density_sweep), np.nan)

    for j, dens in enumerate(density_sweep):
        C_d = build_sparse_coupling(N, dens, coupling_scale,
                                    coupling_seed, symmetric_C)
        actual_dens[j] = np.count_nonzero(C_d) / C_d.size

        E_line = np.full(len(density_alpha), np.nan)
        for i, a in enumerate(density_alpha):
            r1f, r2f, div = run_coupled_inside(
                r1_on, r2_on, C_d, a, n_steps,
                W_res, W_in, b_res, W_out, b_out, leak, record=True)
            if not div:
                dd = np.linalg.norm(r1f - r2f, axis=1)
                E_line[i] = np.mean(dd[len(dd) // 2:])

        below = np.where(E_line < sync_threshold)[0]
        alpha_c_vs_dens[j] = density_alpha[below[0]] if len(below) else np.nan
        tag = (f"{alpha_c_vs_dens[j]:.4f}" if np.isfinite(alpha_c_vs_dens[j])
               else "no sync in range")
        print(f"  density = {actual_dens[j]:.4f}   alpha_c = {tag}")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(actual_dens, alpha_c_vs_dens, 'o-', color='darkgreen',
            markersize=6, linewidth=1.2)
    ax.set_xlabel('coupling density (fraction of nonzero links in C)')
    ax.set_ylabel(r'synchronization threshold  $\alpha_c$')
    ax.set_title(r'$\alpha_c$ vs channel density  —  sparse C inside tanh')
    ax.set_xscale('log')
    ax.grid(alpha=0.3, which='both')
    plt.tight_layout()
    plt.savefig("lorenz_ESN_sparse_inside_alphac_vs_density.png", dpi=120)
    plt.show()

    print("\nSummary table:")
    print(f"  {'density':>9}  {'alpha_c':>10}")
    for dn, ac in zip(actual_dens, alpha_c_vs_dens):
        ac_s = f"{ac:.4f}" if np.isfinite(ac) else "   --   "
        print(f"  {dn:>9.4f}  {ac_s:>10}")